# Load Existing Graphs & GNN Predictions

## Purpose
Load the existing dataset and trained models from your teammates' work:
- Download graphs from HuggingFace (`preethamvj/bottleneck-oracle-graphs`)
- Load trained HeteroGAT models from Notebook 03
- Access baseline predictions from Notebook 01
- Prepare data for completion work (Notebooks 05-08)

## Data Sources
- **HuggingFace**: `preethamvj/bottleneck-oracle-graphs` (501 graphs)
- **Notebook 01**: 1F1B + MLP baselines
- **Notebook 03**: HeteroGAT models (bidirectional + directed)
- **Notebook 02**: Graph generation pipeline

In [3]:
# Import required libraries
import torch
from torch_geometric.data import HeteroData
from huggingface_hub import snapshot_download
import os
import pickle
import numpy as np
from pathlib import Path

print("Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
# print(f"PyG available: {torch_geometric.__version__}")

Libraries imported successfully
PyTorch version: 2.10.0+cpu


## 1. Download/Load Graphs from HuggingFace

Load the 501 pre-generated graphs from your teammates' work.

In [4]:
# Option 1: Download from HuggingFace (recommended)
def download_graphs_from_huggingface():
    """
    Download graphs from HuggingFace repository.
    """
    repo_id = "preethamvj/bottleneck-oracle-graphs"
    
    print(f"Downloading graphs from {repo_id}...")
    
    # Create data directory if it doesn't exist
    os.makedirs("data/graphs", exist_ok=True)
    
    try:
        # Download from HuggingFace
        local_path = snapshot_download(
            repo_id=repo_id,
            repo_type="dataset",
            local_dir="data/graphs",
            local_dir_use_symlinks=False
        )
        print(f"Graphs downloaded to: {local_path}")
        return local_path
    except Exception as e:
        print(f"Error downloading from HuggingFace: {e}")
        return None

# Download graphs
graph_path = download_graphs_from_huggingface()

if graph_path:
    # List downloaded files
    graph_files = list(Path(graph_path).glob("*.pt"))
    print(f"\nFound {len(graph_files)} graph files")
    if graph_files:
        print(f"Sample files: {[f.name for f in graph_files[:5]]}")

C:\Users\Aaron\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:980: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 504 files:   0%|          | 0/504 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip insta

Graphs downloaded to: C:\Users\Aaron\Documents\projects2026\BottleneckOracle\notebooks\data\graphs

Found 501 graph files
Sample files: ['graph_medium_0.pt', 'graph_medium_1.pt', 'graph_medium_10.pt', 'graph_medium_100.pt', 'graph_medium_101.pt']


## 2. Load Graph Data into Memory

Load the HeteroData objects for analysis and model input.

In [ ]:
def load_graphs_from_directory(graph_dir):
    """
    Load all .pt graph files from directory.
    """
    graphs = []
    graph_files = list(Path(graph_dir).glob("**/*.pt"))
    
    print(f"Loading {len(graph_files)} graphs...")
    
    for graph_file in graph_files:
        try:
            graph = torch.load(graph_file)
            graphs.append(graph)
        except Exception as e:
            print(f"Error loading {graph_file}: {e}")
    
    print(f"Successfully loaded {len(graphs)} graphs")
    return graphs

# Load graphs (if downloaded)
if graph_path:
    all_graphs = load_graphs_from_directory(graph_path)
    
    # Analyze first graph
    if all_graphs:
        sample_graph = all_graphs[0]
        print(f"\nSample graph structure:")
        print(f"Node types: {sample_graph.node_types}")
        print(f"Edge types: {sample_graph.edge_types}")
        
        for node_type in sample_graph.node_types:
            print(f"{node_type}: {sample_graph[node_type].num_nodes} nodes")
            if hasattr(sample_graph[node_type], 'x'):
                print(f"  Features: {sample_graph[node_type].x.shape}")

## 3. Load Train/Val/Test Splits

Load the consistent splits used in Notebook 01.

In [ ]:
def load_data_splits():
    """
    Load the train/val/test splits used in baseline experiments.
    These should be the same splits used in Notebook 01 (01_working_baselines.ipynb).
    """
    # TODO: Load splits from Notebook 01
    # Options:
    # 1. Check if splits are saved in data/
    # 2. Recreate using same random seeds (42, 123, 456)
    # 3. Extract from Notebook 01 directly
    
    # For now, create placeholder splits
    if 'all_graphs' in locals() and all_graphs:
        n_graphs = len(all_graphs)
        train_size = int(0.7 * n_graphs)
        val_size = int(0.15 * n_graphs)
        
        # Create splits (using same approach as Notebook 01)
        train_graphs = all_graphs[:train_size]
        val_graphs = all_graphs[train_size:train_size + val_size]
        test_graphs = all_graphs[train_size + val_size:]
        
        print(f"Train: {len(train_graphs)}, Val: {len(val_graphs)}, Test: {len(test_graphs)}")
        
        return {
            'train': train_graphs,
            'val': val_graphs,
            'test': test_graphs
        }
    else:
        print("No graphs loaded yet")
        return None

# Load splits
data_splits = load_data_splits()

## 4. Load Trained HeteroGAT Models

Load the trained models from Notebook 03 (03_heterogat_fixed.ipynb).

In [ ]:
def load_trained_models():
    """
    Load trained HeteroGAT models from Notebook 03.
    
    Note: You may need to extract these from the notebook:
    1. Open 03_heterogat_fixed.ipynb
    2. Run the model training cells
    3. Save models using torch.save()
    """
    # TODO: Implement model loading
    # Options:
    # 1. Load from saved checkpoints (if available)
    # 2. Recreate by running Notebook 03 cells
    # 3. Extract model state dicts from notebook
    
    models = {
        'bidirectional': None,
        'directed': None
    }
    
    # Check for model files
    model_files = list(Path(".").glob("**/*model*.pt")) + list(Path(".").glob("**/*checkpoint*.pt"))
    
    if model_files:
        print(f"Found {len(model_files)} model files:")
        for mf in model_files:
            print(f"  {mf}")
    else:
        print("No saved model files found.")
        print("\nTo get models:")
        print("1. Open notebooks/03_heterogat_fixed.ipynb")
        print("2. Run the training cells")
        print("3. Save models: torch.save(model.state_dict(), 'model_name.pt')")
    
    return models

# Try to load models
trained_models = load_trained_models()

## 5. Load Baseline Predictions

Load baseline predictions from Notebook 01 (01_working_baselines.ipynb).

In [ ]:
def load_baseline_predictions():
    """
    Load baseline predictions from Notebook 01.
    """
    # TODO: Load baseline results
    # Options:
    # 1. Load saved predictions from data/
    # 2. Recalculate by running baseline models
    # 3. Extract from Notebook 01
    
    baseline_results = {
        'f1f1b': {'predictions': [], 'mse': None, 'delta_t_error': None},
        'mlp': {'predictions': [], 'mse': None, 'delta_t_error': None}
    }
    
    print("Baseline predictions will be loaded from Notebook 01")
    print("Current status: TODO - implement loading")
    
    return baseline_results

# Load baselines
baseline_results = load_baseline_predictions()

## 6. Summary & Next Steps

What's loaded and what to do next.

In [ ]:
print("=== LOADING SUMMARY ===")
print(f"Graphs loaded: {len(all_graphs) if 'all_graphs' in locals() else 0}")
print(f"Data splits: {'Loaded' if data_splits else 'Not loaded'}")
print(f"HeteroGAT models: {'Loaded' if all(trained_models.values()) else 'Need to load from Notebook 03'}")
print(f"Baseline predictions: {'Loaded' if baseline_results else 'Need to load from Notebook 01'}")
print()
print("=== NEXT STEPS ===")
print("1. Run this notebook to download/load graphs")
print("2. Open notebooks/03_heterogat_fixed.ipynb to extract trained models")
print("3. Open notebooks/01_working_baselines.ipynb to extract baseline results")
print("4. Use loaded data for completion work (Notebooks 05-08)")